# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets and their fields.

We enumerate all record sets and display their `@id`, name, and a few associated fields/columns (referenced by their `@id`).

In [ ]:
# List all record sets in the dataset
record_sets = list(dataset.record_sets)

print(f"Number of record sets: {len(record_sets)}\n")
all_record_set_ids = []
for rset in record_sets:
    print(f"RecordSet @id: {rset['@id']}")
    all_record_set_ids.append(rset['@id'])
    name = rset.get('name', '(no name)')
    print(f"  Name: {name}")
    if 'field' in rset:
        rset_fields = rset['field'] if isinstance(rset['field'], list) else [rset['field']]
        print(f"  Fields: {[f['@id'] for f in rset_fields]}")
    if 'column' in rset:
        rset_columns = rset['column'] if isinstance(rset['column'], list) else [rset['column']]
        print(f"  Columns: {[c['@id'] for c in rset_columns]}")
    print()
# Display detailed field information for the first record set
if record_sets:
    first_rset = record_sets[0]
    print(f"Fields and columns for RecordSet @id={first_rset['@id']}:")
    if 'field' in first_rset:
        rset_fields = first_rset['field'] if isinstance(first_rset['field'], list) else [first_rset['field']]
        for fld in rset_fields:
            print(f"  Field @id: {fld['@id']}  Name: {fld.get('name', '(no name)')}")
    if 'column' in first_rset:
        rset_columns = first_rset['column'] if isinstance(first_rset['column'], list) else [first_rset['column']]
        for col in rset_columns:
            print(f"  Column @id: {col['@id']}  Name: {col.get('name', '(no name)')}")

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis.
All references to fields and record sets use the `@id` as specified in the schema.

In [ ]:
# We'll construct a dict of DataFrames for each record set by @id

# Safety: reduce to a smaller listed set if large dataset
selected_record_set_ids = all_record_set_ids

dataframes = {}
for record_set_id in selected_record_set_ids:
    # Use records(record_set=...) using the @id of the record set
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}.")
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

# Show columns of the first successfully loaded dataframe
if dataframes:
    example_rset_id = next(iter(dataframes))
    print(f"Columns for record set {example_rset_id}: {dataframes[example_rset_id].columns.tolist()}")
    display(dataframes[example_rset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing: filtering, normalization, and group analysis based on field `@id`s. Operate on the example record set loaded above.

In [ ]:
# Pick a numeric field for demonstration; you can adjust the field_id below

if dataframes:
    # Use the first record set and numeric column as example
    record_set_id = example_rset_id
    df = dataframes[record_set_id]
    numeric_candidates = df.select_dtypes(include=['int', 'float']).columns
    if len(numeric_candidates) > 0:
        numeric_field_id = numeric_candidates[0]  # Use column name (which should be the field @id)
        print(f"Using numeric field @id: {numeric_field_id}")
    else:
        print("No numeric fields found for EDA.")
        numeric_field_id = None

    # Filtering: e.g., select records where value > threshold
    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records: {len(filtered_df)} with {numeric_field_id} > {threshold}")
        display(filtered_df.head())

        # Normalize the numeric column
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized values for {numeric_field_id}: (showing first 5)")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by another field (such as categorical: pick the first object type not already used)
        group_candidates = [col for col in df.select_dtypes(include=['object']).columns if col != numeric_field_id]
        if len(group_candidates) > 0:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id} (showing first 5):")
            display(grouped_df.head())
        else:
            print("No suitable group field for groupby analysis.")
    else:
        print("No numeric field available for filtering and normalization.")

## 5. Visualization
Visualize the distribution of the selected numeric field and (optionally) the grouped analysis (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize field distribution
if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of Field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If a group-by chart was possible
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) FAIR^2 dataset using the `mlcroissant` library.

We loaded structured records using the Croissant schema, identified key record sets and fields via their `@id`, extracted data to DataFrames, and demonstrated EDA workflows including filtering, normalization, and grouping. The workflow is reproducible and adaptable: update the field and record set `@id`s for further analysis across the dataset.

For additional exploration, consult the dataset schema for all available `@id`s. For more examples and documentation, visit the [mlcroissant documentation](https://mlcommons.github.io/croissant/).